In [ ]:
import os, shutil, glob, subprocess
from google.colab import drive

drive.mount('/content/drive')

# Clone gaussian-splatting
if not os.path.exists('/content/gaussian-splatting/train.py'):
    if os.path.exists('/content/gaussian-splatting'):
        shutil.rmtree('/content/gaussian-splatting')
    subprocess.run(["git", "clone", "--recursive",
        "https://github.com/graphdeco-inria/gaussian-splatting.git",
        "/content/gaussian-splatting"], check=True)
    subprocess.run(["pip", "install", "plyfile", "tqdm", "-q"])
    os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0"
    for ext in ["diff-gaussian-rasterization", "simple-knn"]:
        r = subprocess.run(["pip", "install", "-e", "."],
                           cwd=f"/content/gaussian-splatting/submodules/{ext}",
                           capture_output=True, text=True)
        print(f"{ext}: {'OK' if r.returncode==0 else r.stderr[-200:]}")

# Install COLMAP and pycolmap
subprocess.run(["apt-get", "install", "-y", "-q", "colmap"], check=True)
subprocess.run(["pip", "install", "pycolmap", "pillow-heif", "-q"])
print("=== INSTALL COMPLETE ===")

Mounted at /content/drive
diff-gaussian-rasterization: OK
simple-knn: OK
=== INSTALL COMPLETE ===


In [ ]:
from pillow_heif import register_heif_opener
from PIL import Image
import os

register_heif_opener()

src = '/content/drive/MyDrive/living_room_nerf'
dst_gs = '/content/gaussian-splatting/data/living_room/input'
os.makedirs(dst_gs, exist_ok=True)

if len(os.listdir(dst_gs)) < 10:
    converted = 0
    for f in os.listdir(src):
        fp = os.path.join(src, f)
        out_name = f.replace('.HEIC', '.JPG')
        out_path = os.path.join(dst_gs, out_name)
        if f.endswith(('.JPG', '.HEIC')):
            img = Image.open(fp).resize((800, 600), Image.LANCZOS)
            img.save(out_path, "JPEG", quality=90)
            converted += 1
    print(f"Converted {converted} images")
else:
    print(f"Images already present: {len(os.listdir(dst_gs))}")

Converted 263 images


In [ ]:
import pycolmap, os, shutil

gs_data = "/content/gaussian-splatting/data/living_room"

# Clean any previous COLMAP run
for p in ["colmap.db", "sparse"]:
    fp = f"{gs_data}/{p}"
    if os.path.isfile(fp): os.remove(fp)
    elif os.path.isdir(fp): shutil.rmtree(fp)

os.makedirs(f"{gs_data}/sparse", exist_ok=True)

reader_options = pycolmap.ImageReaderOptions()
reader_options.camera_model = "SIMPLE_PINHOLE"

print("Extracting features...")
pycolmap.extract_features(
    f"{gs_data}/colmap.db",
    f"{gs_data}/input",
    reader_options=reader_options
)
print("Matching (this takes 20-40 min)...")
pycolmap.match_exhaustive(f"{gs_data}/colmap.db")
print("Mapping...")
maps = pycolmap.incremental_mapping(
    f"{gs_data}/colmap.db",
    f"{gs_data}/input",
    f"{gs_data}/sparse"
)
print(f"Registered: {len(list(maps[0].images.values()))} / {len(os.listdir(gs_data+'/input'))} images")

Extracting features...
Matching (this takes 20-40 min)...
Mapping...
Registered: 241 / 263 images


In [ ]:
recon = maps[0]
bad = []
for img_id, img in recon.images.items():
    cam = recon.cameras[img.camera_id]
    w, h = cam.width, cam.height
    if w == 0 or h == 0 or w > 10000 or h > 10000:
        bad.append((img_id, w, h))

if bad:
    print(f"⚠️  BAD CAMERAS FOUND: {bad}")
    print("Do NOT proceed — re-run Cell 3")
else:
    print(f"✅ All {len(recon.cameras)} cameras valid")
    for cam in recon.cameras.values():
        print(f"   Camera: {cam.width}x{cam.height}, model={cam.model_name}")

✅ All 241 cameras valid
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_PINHOLE
   Camera: 800x600, model=SIMPLE_

In [ ]:
import subprocess, os, shutil

gs_data = "/content/gaussian-splatting/data/living_room"

if os.path.exists(f"{gs_data}/undistorted"):
    shutil.rmtree(f"{gs_data}/undistorted")
os.makedirs(f"{gs_data}/undistorted", exist_ok=True)

result = subprocess.run([
    "colmap", "image_undistorter",
    "--image_path",     f"{gs_data}/input",
    "--input_path",     f"{gs_data}/sparse/0",
    "--output_path",    f"{gs_data}/undistorted",
    "--output_type",    "COLMAP",
    "--max_image_size", "800"
], capture_output=True, text=True)

print(result.stdout[-1000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
else:
    print("✅ Undistortion complete")
    print("Contents:", os.listdir(f"{gs_data}/undistorted"))
    print("Sparse:",   os.listdir(f"{gs_data}/undistorted/sparse"))


Undistorting image [211/241]
Undistorting image [212/241]
Undistorting image [213/241]
Undistorting image [214/241]
Undistorting image [215/241]
Undistorting image [216/241]
Undistorting image [217/241]
Undistorting image [218/241]
Undistorting image [219/241]
Undistorting image [220/241]
Undistorting image [221/241]
Undistorting image [222/241]
Undistorting image [223/241]
Undistorting image [224/241]
Undistorting image [225/241]
Undistorting image [226/241]
Undistorting image [227/241]
Undistorting image [228/241]
Undistorting image [229/241]
Undistorting image [230/241]
Undistorting image [231/241]
Undistorting image [232/241]
Undistorting image [233/241]
Undistorting image [234/241]
Undistorting image [235/241]
Undistorting image [236/241]
Undistorting image [237/241]
Undistorting image [238/241]
Undistorting image [239/241]
Undistorting image [240/241]
Undistorting image [241/241]
Writing reconstruction...
Writing configuration...
Writing scripts...
Elapsed time: 0.049 [minutes]


In [ ]:
import subprocess, os

gs_data = "/content/gaussian-splatting/data/living_room"
sparse_path = f"{gs_data}/undistorted/sparse"

os.makedirs(f"{sparse_path}/0", exist_ok=True)
result = subprocess.run([
    "colmap", "model_converter",
    "--input_path",  sparse_path,
    "--output_path", f"{sparse_path}/0",
    "--output_type", "TXT"
], capture_output=True, text=True)

if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("✅ Converted to TXT")

cam_file = f"{sparse_path}/0/cameras.txt"
with open(cam_file) as f:
    for line in f:
        if line.startswith("#") or not line.strip():
            continue
        parts = line.split()
        w, h = int(parts[2]), int(parts[3])
        print(f"Camera: {w}x{h}, params: {parts[4:]}")
        if w == 0 or h == 0:
            raise ValueError(f"❌ Zero-dimension camera W={w} H={h} — re-run Cell 3")

print("✅ Cameras look good, safe to proceed")

✅ Converted to TXT
Camera: 800x600, params: ['574.2033708725279', '574.2033708725279', '400', '300']
Camera: 800x600, params: ['568.21514221887935', '568.21514221887935', '400', '300']
Camera: 800x600, params: ['570.98017837654982', '570.98017837654982', '400', '300']
Camera: 800x600, params: ['572.15731762499672', '572.15731762499672', '400', '300']
Camera: 800x600, params: ['558.70683295295612', '558.70683295295612', '400', '300']
Camera: 800x600, params: ['566.13690421894569', '566.13690421894569', '400', '300']
Camera: 800x600, params: ['565.64929788186441', '565.64929788186441', '400', '300']
Camera: 800x600, params: ['562.72197653641524', '562.72197653641524', '400', '300']
Camera: 800x600, params: ['565.09571767957493', '565.09571767957493', '400', '300']
Camera: 800x600, params: ['564.30578324970088', '564.30578324970088', '400', '300']
Camera: 800x600, params: ['565.3428950319194', '565.3428950319194', '400', '300']
Camera: 800x600, params: ['568.43752596128047', '568.43752596

In [ ]:
import os, shutil

gs_data = "/content/gaussian-splatting/data/living_room"

src_images = f"{gs_data}/undistorted/images"
dst_images = f"{gs_data}/undistorted_small/images"
os.makedirs(dst_images, exist_ok=True)

all_imgs = sorted(os.listdir(src_images))
kept = all_imgs[::3]  # every 3rd image
print(f"Keeping {len(kept)} / {len(all_imgs)} images")

for f in kept:
    shutil.copy(f"{src_images}/{f}", f"{dst_images}/{f}")

shutil.copytree(
    f"{gs_data}/undistorted/sparse",
    f"{gs_data}/undistorted_small/sparse",
    dirs_exist_ok=True
)
print("✅ Subsampled dataset ready")

Keeping 81 / 241 images
✅ Subsampled dataset ready


In [ ]:
# Only patch needed on A100 - just a safety net, probably won't trigger
model_file = "/content/gaussian-splatting/scene/gaussian_model.py"
txt = open(model_file).read()

old = "        scales = torch.log(torch.sqrt(dist2))[...,None].repeat(1, 3)"
new = "        scales = torch.log(torch.sqrt(dist2.cuda()))[...,None].repeat(1, 3)"

if old in txt:
    open(model_file, 'w').write(txt.replace(old, new))
    print("✅ scales patch applied")
else:
    print("⚠️ Already patched or pattern changed")

✅ scales patch applied


In [ ]:
import subprocess, os

gs_data = "/content/gaussian-splatting/data/living_room"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

process = subprocess.Popen([
    "python", "train.py",
    "-s", f"{gs_data}/undistorted",        # ← full 241 cameras
    "-m", "output/living_room",
    "--iterations",             "30000",    # ← full training
    "--resolution",             "2",        # ← good quality, safe on A100
    "--densify_grad_threshold", "0.0002",
    "--densification_interval", "100",      # ← more frequent = better quality
    "--opacity_reset_interval", "3000",
    "--position_lr_max_steps",  "30000",
    "--save_iterations",        "7000", "15000", "30000",
    "--test_iterations",        "-1",
], cwd="/content/gaussian-splatting",
   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

process.wait()
print(f"\nReturn code: {process.returncode}")

Streaming output truncated to the last 5000 lines.
Training progress: 100%|██████████| 30000/30000 [10:48<00:00, 46.28it/s, Loss=0.0212148, Depth Loss=0.0000000]

[ITER 7000] Saving Gaussians [25/03 00:29:23]

[ITER 15000] Saving Gaussians [25/03 00:32:41]

[ITER 30000] Saving Gaussians [25/03 00:38:29]

Training complete. [25/03 00:38:50]

Return code: 0


In [ ]:
import shutil, os

out_src = "/content/gaussian-splatting/output/living_room"
out_dst = "/content/drive/MyDrive/living_room_nerf/gs_output"

if os.path.exists(out_dst):
    shutil.rmtree(out_dst)
shutil.copytree(out_src, out_dst)
print(f"✅ Saved to Drive: {out_dst}")
print("Files:", os.listdir(out_dst))

✅ Saved to Drive: /content/drive/MyDrive/living_room_nerf/gs_output
Files: ['cameras.json', 'point_cloud', 'exposure.json', 'input.ply', 'cfg_args', 'events.out.tfevents.1774398451.e6cf9b8b7883.21781.0']


In [ ]:
import os

out = "/content/drive/MyDrive/living_room_nerf/gs_output"
for root, dirs, files in os.walk(f"{out}/point_cloud"):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"{path} ({size:.1f} MB)")

/content/drive/MyDrive/living_room_nerf/gs_output/point_cloud/iteration_15000/point_cloud.ply (531.9 MB)
/content/drive/MyDrive/living_room_nerf/gs_output/point_cloud/iteration_7000/point_cloud.ply (436.7 MB)
/content/drive/MyDrive/living_room_nerf/gs_output/point_cloud/iteration_30000/point_cloud.ply (531.9 MB)


In [ ]:
# Compress the point cloud for Jetson Nano deployment
import subprocess, os

ply_src = "/content/drive/MyDrive/living_room_nerf/gs_output/point_cloud/iteration_30000/point_cloud.ply"
ply_out = "/content/drive/MyDrive/living_room_nerf/gs_output/point_cloud_nano.ply"

# Install and run gsplat compression
subprocess.run(["pip", "install", "plyfile", "numpy", "-q"])

import numpy as np
from plyfile import PlyData, PlyElement

print("Loading PLY...")
plydata = PlyData.read(ply_src)
vertex = plydata['vertex']
print(f"Original Gaussians: {len(vertex):,}")

# Sort by opacity and keep top N
# opacity is stored as 'opacity' field in the ply
opacity = np.array(vertex['opacity'])
# sigmoid to get actual opacity
opacity_sigmoid = 1 / (1 + np.exp(-opacity))

# Keep only Gaussians above opacity threshold
threshold = 0.1
mask = opacity_sigmoid > threshold
print(f"Gaussians above opacity {threshold}: {mask.sum():,}")

# Further limit to top 300k for Jetson
max_gaussians = 300_000
if mask.sum() > max_gaussians:
    opacities_filtered = opacity_sigmoid[mask]
    top_idx = np.argsort(opacities_filtered)[-max_gaussians:]
    indices = np.where(mask)[0][top_idx]
    mask = np.zeros(len(vertex), dtype=bool)
    mask[indices] = True

print(f"Final Gaussians for Nano: {mask.sum():,}")

# Write compressed PLY
filtered = PlyElement.describe(vertex.data[mask], 'vertex')
PlyData([filtered]).write(ply_out)

size = os.path.getsize(ply_out) / 1024 / 1024
print(f"✅ Compressed PLY saved: {size:.1f} MB")
print(f"   Saved to: {ply_out}")

Loading PLY...
Original Gaussians: 2,249,028
Gaussians above opacity 0.1: 794,384
Final Gaussians for Nano: 300,000
✅ Compressed PLY saved: 71.0 MB
   Saved to: /content/drive/MyDrive/living_room_nerf/gs_output/point_cloud_nano.ply
